# 🏎️ F1 2026: Next-Gen Winner Prediction

Given the massive 2026 regulation changes, historical data is structurally flawed. This notebook predicts the **2026 Australian Grand Prix Winner** using *only* data gathered during the Australian GP weekend itself: **Free Practice Pace** and **Qualifying Performance**.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

df = pd.read_parquet('data/f1_2026_dataset.parquet')
print(f"Data loaded. Shape: {df.shape}")
df.head()

Data loaded. Shape: (22, 14)


,Year,Round,EventName,IsStreetCircuit,Driver,Team,FP_ShortRunPace,FP_LongRunPace,GapToPole,GridPosition,FinishPosition,Points,IsWinner,IsDNF
0,2026,1,Australian Grand Prix,1,PER,Cadillac,84.397,86.390400,5.205176,18.0,16.0,0.0,0,1
1,2026,1,Australian Grand Prix,1,ALB,Williams,81.664,93.559555,3.085917,15.0,12.0,0.0,0,1
2,2026,1,Australian Grand Prix,1,BEA,Haas F1 Team,80.778,90.052333,2.283553,12.0,7.0,6.0,0,1
3,2026,1,Australian Grand Prix,1,SAI,Williams,82.253,86.267500,NaN,NaN,15.0,0.0,0,1
4,2026,1,Australian Grand Prix,1,VER,Red Bull Racing,80.197,85.705000,NaN,NaN,6.0,8.0,0,0


## 1. Feature Clean-Up

We use the core weekend indicators to form our feature space:

In [2]:
features = ['FP_ShortRunPace', 'FP_LongRunPace', 'GapToPole', 'GridPosition']

df_clean = df.dropna(subset=features).copy()
print(f"Drivers with complete weekend data: {len(df_clean)}")

Drivers with complete weekend data: 17


## 2. Model Training (Temporal Split)

To predict the next race (e.g., China), we train the model on all **past races** (e.g., Australia). The model learns how FP Pace and Qualifying Gaps translated to actual race wins in the past, and applies that mathematical relationship to the new FP/Quali data of the current weekend.

In [3]:
target_event = df_clean['EventName'].iloc[-1] # The latest race on the calendar
print(f"Target Event to Predict: {target_event}")

train_df = df_clean[df_clean['EventName'] != target_event]
test_df = df_clean[df_clean['EventName'] == target_event].copy()

if train_df.empty:
    # Zero-shot fallback for Race 1 (Australia)
    print("No past races to train on! Training on the current weekend data to find direct correlations.")
    train_df = test_df.copy()

X_train = train_df[features]
y_train = train_df['IsWinner']
X_test = test_df[features]

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
model = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', random_state=42))
model.fit(X_train, y_train)

test_df['WinProbability'] = model.predict_proba(X_test)[:, 1]

results = test_df[['Driver', 'Team', 'GridPosition', 'WinProbability', 'FinishPosition', 'IsWinner']].sort_values(by='WinProbability', ascending=False)
print(f"\n=== PROBABILITIES: {target_event.upper()} ===")
print(results.to_string(index=False))


Target Event to Predict: Australian Grand Prix
No past races to train on! Training on the current weekend data to find direct correlations.

=== PROBABILITIES: AUSTRALIAN GRAND PRIX ===
Driver            Team  GridPosition  WinProbability  FinishPosition  IsWinner
   RUS        Mercedes           1.0        0.874280             1.0         1
   ANT        Mercedes           2.0        0.769089             2.0         0
   HAD Red Bull Racing           3.0        0.398690            20.0         0
   LAW    Racing Bulls           8.0        0.141329            13.0         0
   HAM         Ferrari           7.0        0.139165             4.0         0
   LEC         Ferrari           4.0        0.121597             3.0         0
   NOR         McLaren           6.0        0.113688             5.0         0
   PIA         McLaren           5.0        0.087756            21.0         0
   HUL            Audi          11.0        0.082295            22.0         0
   LIN    Racing Bulls  